In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.svm import SVC

In [3]:
df = pd.read_csv('loan_data.csv')
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [4]:
X = df.drop(columns = 'loan_status')
y= df.loan_status

In [5]:
xtrain, xtest, ytrain, ytest = train_test_split(X,y,train_size=0.8, random_state=42)

In [6]:
#num_cols = X.select_dtypes(include='number').columns
obj_cols = X.select_dtypes(include='object').columns

C:\Users\DELL\AppData\Local\Temp\ipykernel_1436\1433802163.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = X.select_dtypes(include='object').columns


In [7]:
X[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [8]:
# person_gender                     2
# person_education                  5
# person_home_ownership             4
# loan_intent                       6
# previous_loan_defaults_on_file    2
# 2 unique values and less feature values do OneHotEncoding and more unique values do OrdinalEncoding

In [9]:
X['person_education'].unique()

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str

In [10]:
order = ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [11]:
obj_cols.drop(['person_education'])

Index(['person_gender', 'person_home_ownership', 'loan_intent',
       'previous_loan_defaults_on_file'],
      dtype='str')

In [12]:
# preprocessing = ColumnTransformer(
#     transformers=[
#         ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'),obj_cols.drop('person_education')),
#         ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),['person_education']))
#     ],remainder='passthrough'
# )

# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing', preprocessing),
#         ('model', DecisionTreeClassifier(random_state=42))
#     ]
# )

# grid_search_cv = GridSearchCV(
#     estimator = main_pipeline,
#     param_grid = {
#         'model__criterion':['gini','entropy'], #model__ is used to access the parameters of the model in the pipeline i.e go inside model and access the parameters of DecisionTreeClassifier
#         'model__max_depth':[None,5,10,50,100], #if we use algorithm no need to use model__ but if we use pipeline then we need to use model__ to access the parameters of the model in the pipeline
#         'model__min_samples_split':[2,5,7,10],
#         'model__min_samples_leaf':[1,3,5,7,10],
#         'model__splitter':['best', 'random']
#     },
# verbose=1)
# grid_search_cv.fit(xtrain,ytrain)

In [13]:
preprocessing = ColumnTransformer(
    transformers=[
        ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'), obj_cols.drop('person_education')),
        ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1), ['person_education'])
    ],
    remainder='passthrough'
)

main_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

grid_search_cv = GridSearchCV(
    estimator = main_pipeline,
    param_grid = {
        'model__criterion':['gini','entropy'],
        'model__max_depth':[None,5,10,50,100],
        'model__min_samples_split':[2,5,7,10],
        'model__min_samples_leaf':[1,3,5,7,10],
        'model__splitter':['best', 'random']
    },
    verbose=1,n_jobs=-1
)
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 400 candidates, totalling 2000 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the c

In [14]:
# preprocessing = ColumnTransformer(
#     transformers=[
#         ('onehot_encoder', OneHotEncoder(handle_unknown='ignore'), obj_cols.drop('person_education')),
#         ('ordinal_encoder', OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1), ['person_education'])
#     ],
#     remainder='passthrough'
# )

# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing', preprocessing),
#         ('model', SVC(random_state=42))
#     ]
# )

# grid_search_cv = GridSearchCV(
#     estimator = main_pipeline,
#     param_grid = {
#         'model__C':[0.01,0.1,1.0,10,100],
#         'model__kernel':['linear', 'poly', 'rbf', 'sigmoid']
#     },
#     verbose=1,n_jobs=-1
# )
# grid_search_cv.fit(xtrain,ytrain)

In [15]:
grid_search_cv.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot_encoder', ...), ('ordinal_encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output o

In [16]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [17]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.62401962, 0.44465837, 0.73099871, 0.6983994 , 0.79343023,
        0.45411286, 0.64652166, 0.43809261, 0.68398485, 0.44920402,
        0.73818588, 0.4783627 , 0.66057367, 0.44541316, 0.68267016,
        0.45777583, 0.64593234, 0.46960669, 0.67861233, 0.45823407,
        0.67390828, 0.45645995, 0.65602751, 0.42926211, 0.62202988,
        0.49789562, 0.77058749, 0.474369  , 0.8702281 , 0.48980885,
        0.66632776, 0.46597562, 0.65895085, 0.4746489 , 0.69326057,
        0.52847714, 0.66124949, 0.41087017, 0.6622745 , 0.48336554,
        0.5205092 , 0.42801995, 0.52059178, 0.43562551, 0.52187715,
        0.48609886, 0.6246335 , 0.49412103, 0.53157253, 0.47033339,
        0.57310295, 0.49226327, 0.50252652, 0.43869176, 0.51519494,
        0.47781806, 0.50752826, 0.51223741, 0.61367421, 0.47702861,
        0.75250015, 0.66904058, 0.68322754, 0.57874999, 0.58675575,
        0.64599013, 0.63317428, 0.43741617, 0.56336937, 0.50771012,
        0.59326158, 0.53478861,

In [18]:
results = pd.DataFrame(grid_search_cv.cv_results_)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.624020,0.075974,0.070561,0.014082,gini,None,1,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.896806,0.900972,0.899444,0.901250,0.895694,0.898833,0.002225,316
1,0.444658,0.090550,0.066948,0.010090,gini,None,1,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.888333,0.891111,0.884583,0.892639,0.889722,0.889278,0.002749,355
2,0.730999,0.118159,0.074535,0.016618,gini,None,1,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.895139,0.901806,0.900000,0.901667,0.901250,0.899972,0.002499,301
3,0.698399,0.076076,0.085396,0.024325,gini,None,1,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.895278,0.897778,0.890000,0.889306,0.895417,0.893556,0.003315,349
4,0.793430,0.100011,0.066736,0.005385,gini,None,1,7,best,"{'model__criterion': 'gini', 'model__max_depth...",0.897917,0.900556,0.900556,0.903472,0.901667,0.900833,0.001807,292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,0.361105,0.035460,0.066231,0.012339,entropy,100,10,5,random,"{'model__criterion': 'entropy', 'model__max_de...",0.902222,0.909444,0.899861,0.904583,0.909722,0.905167,0.003904,186
396,0.685344,0.030004,0.067016,0.016158,entropy,100,10,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908194,0.915139,0.912778,0.912917,0.912639,0.912333,0.002264,81
397,0.382556,0.061048,0.074415,0.016030,entropy,100,10,7,random,"{'model__criterion': 'entropy', 'model__max_de...",0.902222,0.909444,0.899861,0.904583,0.909722,0.905167,0.003904,186
398,0.623603,0.014607,0.058777,0.008972,entropy,100,10,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.908194,0.915139,0.912778,0.912917,0.912639,0.912333,0.002264,81


In [19]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
302,0.661985,0.098832,0.066770,0.020205,entropy,10,5,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
300,0.729477,0.148434,0.102742,0.033349,entropy,10,5,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
298,0.677057,0.151657,0.094873,0.029003,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
280,0.644754,0.101314,0.067393,0.016868,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.926111,0.914861,0.920833,0.920556,0.920389,0.003583,1
296,0.705295,0.131775,0.070867,0.011404,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0.534789,0.127804,0.068511,0.020393,gini,5,7,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872639,0.872417,0.000461,393
77,0.800707,0.274807,0.112988,0.033499,gini,5,10,7,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
79,0.607669,0.170934,0.074963,0.015774,gini,5,10,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
73,0.598810,0.097369,0.126181,0.028210,gini,5,10,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397


In [20]:
#do the same with logistic regression and random forest 